In [1]:
# =====================================
# CONFIGURATION
# =====================================

import pandas as pd
import numpy as np
import math
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc
)

RANDOM_STATE = 42
ANOMALY_CONTAMINATION = 0.01
RISK_THRESHOLD = 0.65

In [2]:
# =====================================
# DATA LOADING
# =====================================

def load_data(path):

    df = pd.read_csv(path)

    df.columns = df.columns.str.strip()

    df['Transaction Time'] = pd.to_datetime(df['Transaction Time'])

    df = df.sort_values('Transaction Time').reset_index(drop=True)

    df['FRAUD'] = (df['Purpose'] == "Fraud Transaction").astype(int)

    df['Amount_Spike_Flag'] = (
        df['Amount Spikes']
        .astype(str)
        .str.lower()
        .str.strip()
        .eq("yes")
        .astype(int)
    )

    return df


df = load_data("transactions_mapped.csv")

print("Total Records:", len(df))
print("Fraud Count:", df['FRAUD'].sum())

Total Records: 11609
Fraud Count: 17


In [3]:
# =====================================
# BEHAVIORAL FEATURES
# =====================================

def behavioral_features(df):

    df['hour'] = df['Transaction Time'].dt.hour

    df['is_night'] = df['hour'].between(0,5).astype(int)

    df['time_gap_sec'] = (
        df['Transaction Time']
        .diff()
        .dt.total_seconds()
        .fillna(0)
    )

    df['prev_balance'] = df['Total Amt'].shift(1).fillna(df['Total Amt'])

    df['balance_drain_pct'] = (
        (df['prev_balance'] - df['Total Amt']) / df['prev_balance']
    )

    df['balance_drain_pct'] = (
        df['balance_drain_pct']
        .replace([np.inf,-np.inf],0)
        .fillna(0)
    )

    return df


df = behavioral_features(df)

In [4]:
# =====================================
# IMPOSSIBLE TRAVEL DETECTION
# =====================================

city_coords = {
"Koramangala": (12.9352, 77.6245),
"Chennai": (13.0827, 80.2707),
"Indiranagar": (12.9784, 77.6408),
"Whitefield": (12.9698, 77.7500),
"HSR Layout": (12.9116, 77.6474),
"MG Road": (12.9758, 77.6046),
"Jayanagar": (12.9250, 77.5938),
"Malleshwaram": (13.0031, 77.5713),
"Delhi": (28.6519, 77.2315),
"Bihar": (25.1948, 85.5170),
"Hyderabad": (17.3871, 78.4917),
"Pune": (18.5167, 73.8563),
"Kochi": (9.9312, 76.2673)
}


def compute_travel(df):

    df['lat'] = df['Location'].map(lambda x: city_coords.get(x,(np.nan,np.nan))[0])
    df['lon'] = df['Location'].map(lambda x: city_coords.get(x,(np.nan,np.nan))[1])

    df['prev_lat'] = df['lat'].shift(1)
    df['prev_lon'] = df['lon'].shift(1)

    R = 6371

    lat1 = np.radians(df['prev_lat'])
    lat2 = np.radians(df['lat'])

    lon1 = np.radians(df['prev_lon'])
    lon2 = np.radians(df['lon'])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2

    c = 2*np.arctan2(np.sqrt(a),np.sqrt(1-a))

    distance = R*c

    df['travel_speed_kmh'] = distance/(df['time_gap_sec']/3600)

    df['travel_speed_kmh'] = df['travel_speed_kmh'].replace([np.inf,-np.inf],0).fillna(0)

    df['impossible_travel_flag'] = (df['travel_speed_kmh'] > 500).astype(int)

    return df


df = compute_travel(df)

In [5]:
# =====================================
# IDENTITY STABILITY
# =====================================

df['ip_change_flag'] = (df['Device IP'] != df['Device IP'].shift(1)).astype(int)

df['location_change_flag'] = (df['Location'] != df['Location'].shift(1)).astype(int)

In [6]:
# =====================================
# VELOCITY FEATURE
# =====================================

df = df.set_index('Transaction Time')

df['velocity_1h'] = df['Debited Amt'].rolling('1H').count()

df = df.reset_index()

C:\Users\nayab\AppData\Local\Temp\ipykernel_7420\2539292118.py:7: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['velocity_1h'] = df['Debited Amt'].rolling('1H').count()


In [7]:
# =====================================
# Z-SCORE FEATURES
# =====================================

normal_df = df[df['FRAUD']==0]


def compute_zscore(column):

    mean = normal_df[column].mean()

    std = normal_df[column].std()

    if std == 0:
        return np.zeros(len(df))

    return (df[column]-mean)/std


df['debit_zscore'] = compute_zscore('Debited Amt')

df['timegap_zscore'] = compute_zscore('time_gap_sec')

df['velocity_zscore'] = compute_zscore('velocity_1h')

In [8]:
# =====================================
# ANOMALY MODEL
# =====================================

features = [
'Debited Amt',
'hour',
'time_gap_sec',
'velocity_1h',
'balance_drain_pct',
'ip_change_flag',
'location_change_flag',
'Amount_Spike_Flag'
]

X_normal = normal_df[features]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_normal)

iso_model = IsolationForest(
    contamination=ANOMALY_CONTAMINATION,
    random_state=RANDOM_STATE
)

iso_model.fit(X_scaled)

X_all = scaler.transform(df[features])

df['anomaly_score'] = -iso_model.decision_function(X_all)

In [9]:
# =====================================
# RULE ENGINE
# =====================================

df['rule_score'] = (
    (df['debit_zscore'] > 3)*40 +
    (df['balance_drain_pct'] > 0.5)*50 +
    (df['is_night'])*10 +
    (df['ip_change_flag'])*20 +
    (df['location_change_flag'])*20 +
    (df['velocity_zscore'] > 3)*30 +
    (df['impossible_travel_flag'])*50
)

In [10]:
# =====================================
# HYBRID RISK SCORE
# =====================================

df['rule_score_norm'] = df['rule_score']/df['rule_score'].max()

df['anomaly_score_norm'] = (
    df['anomaly_score'] - df['anomaly_score'].min()
)/(
    df['anomaly_score'].max() - df['anomaly_score'].min()
)

df['final_risk'] = (
    0.6 * df['anomaly_score_norm'] +
    0.4 * df['rule_score_norm']
)

In [11]:
conditions = [
df['final_risk'] < 0.3,
df['final_risk'] < 0.7
]

choices = ["ALLOW","OTP_REQUIRED"]

df['decision'] = np.select(conditions,choices,default="BLOCK")

In [12]:
df['predicted_fraud'] = (df['final_risk'] >= RISK_THRESHOLD).astype(int)

In [13]:
y_true = df['FRAUD']

y_pred = df['predicted_fraud']

y_prob = df['final_risk']


cm = confusion_matrix(y_true,y_pred)

print("Confusion Matrix:\n",cm)

print(classification_report(y_true,y_pred))

roc_auc = roc_auc_score(y_true,y_prob)

print("ROC-AUC:",roc_auc)

precision,recall,_ = precision_recall_curve(y_true,y_prob)

print("PR-AUC:",auc(recall,precision))

Confusion Matrix:
 [[11501    91]
 [    2    15]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     11592
           1       0.14      0.88      0.24        17

    accuracy                           0.99     11609
   macro avg       0.57      0.94      0.62     11609
weighted avg       1.00      0.99      0.99     11609

ROC-AUC: 0.9957932448341655
PR-AUC: 0.5877542426768468


In [14]:
df.to_csv("processed_transactions.csv", index=False)

In [15]:
import joblib

joblib.dump(iso_model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']